In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns  


import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

# 물류센터A 생수,음료,건강

In [ ]:
df = pd.read_csv('./수요예측부문 데이터 제공/csv파일 모음/A물류센터 매출_2021~2024_생수음료건강_월별판매수량.csv')
df['연월'] = pd.to_datetime(df['연월'].astype(str))
# ===== 데이터 준비 =====
df['연월'] = pd.to_datetime(df['연월'])
df.set_index('연월', inplace=True)
df.index = pd.to_datetime(df.index)
df = df.asfreq('MS')

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(df['판매수량'])
plt.title("Original")

plt.subplot(1,2,2)
plt.hist(df['판매수량'], bins=30)
plt.title("Distribution")
plt.show()

In [ ]:
# acf, pacf

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plot_acf(df['판매수량'], lags=24)
plt.show()
plot_pacf(df['판매수량'], lags=24)
plt.show()

## SARIMA(외생변수X)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings("ignore")

# ===== 데이터 준비 =====

df = df.asfreq('MS')

# ===== 파라미터 설정 =====
season_length = 12   # 월별 데이터니까 12개월 주기
folds = 5          # 교차검증 fold 개수
h = 3                # 한 번에 예측할 기간 (3개월 ahead)

rmse_list = []
pred_df_list = []    # fold별 예측값 저장용

# ===== Rolling CV =====
for i in range(1, folds+1):
    split_point = len(df) - (folds - i)*h
    train = df.iloc[:split_point - h]
    test = df.iloc[split_point - h:split_point]

    if len(train) < season_length:
        continue

    # SARIMA(1,1,1)(0,1,1,12) 모델
    model = SARIMAX(train['판매수량'],
                    order=(1,1,1),
                    seasonal_order=(0,1,1,12),
                    enforce_stationarity=False,
                    enforce_invertibility=False)
    result = model.fit(disp=False)

    # 3개월 ahead 예측
    forecast = result.forecast(steps=h)
    forecast = pd.Series(forecast, index=test.index, name=f'fold{i}_예측')

    rmse = np.sqrt(mean_squared_error(test['판매수량'], forecast))
    rmse_list.append(rmse)
    pred_df_list.append(pd.DataFrame({'실제': test['판매수량'], '예측': forecast}))

    print(f"Fold {i}: Train={train.index[0].strftime('%Y-%m')}~{train.index[-1].strftime('%Y-%m')}, "
          f"Test={test.index[0].strftime('%Y-%m')}~{test.index[-1].strftime('%Y-%m')}, RMSE={rmse:,.0f}")

# ===== 결과 요약 =====
print("\n📊 평균 RMSE:", round(np.mean(rmse_list), 2))

# ===== 시각화 =====
plt.figure(figsize=(12,6))
plt.plot(df.index, df['판매수량'], label='Actual', color='black')

# 각 fold별 예측 구간 시각화
for i, pred in enumerate(pred_df_list):
    plt.plot(pred.index, pred['예측'], label=f'Fold {i+1} Forecast', linestyle='--')

plt.title('SARIMA(1,1,1)(1,1,1,12) Rolling Forecast (3개월 단위, 10-Fold)')
plt.xlabel('날짜')
plt.ylabel('판매수량')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

import warnings
warnings.filterwarnings('ignore')

# ===== 데이터 준비 =====

df = df.asfreq('MS')

# ===== 파라미터 설정 =====
season_length = 12   # 월별 데이터니까 12개월 주기
folds = 5           # 교차검증 fold 개수
h = 3                # 한 번에 예측할 기간 (3개월 ahead)

import itertools

# 후보 파라미터 조합
param_list = [
    ((1,1,1), (1,1,1,12)),
    ((0,1,1), (0,1,1,12)),
    ((1,1,1), (0,1,1,12)),
    ((2,1,1), (1,1,0,12)),
    ((1,1,2), (0,1,1,12)),
    ((2,1,2), (1,1,1,12)),

]

results = []

for (order, seasonal_order) in param_list:
    rmse_list = []
    print(f"\n===== 모델: SARIMA{order}x{seasonal_order} =====")

    for i in range(1, folds+1):
        split_point = len(df) - (folds - i)*h
        train = df.iloc[:split_point - h]
        test = df.iloc[split_point - h:split_point]

        if len(train) < season_length:
            continue

        model = SARIMAX(train['판매수량'],
                        order=order,
                        seasonal_order=seasonal_order,
                        enforce_stationarity=False,
                        enforce_invertibility=False)
        result = model.fit(disp=False)

        forecast = result.forecast(steps=h)
        rmse = np.sqrt(mean_squared_error(test['판매수량'], forecast))
        rmse_list.append(rmse)

    avg_rmse = np.mean(rmse_list)
    results.append((order, seasonal_order, avg_rmse))
    print(f"📊 평균 RMSE = {avg_rmse:,.2f}")

# 결과를 표로 정리
results_df = pd.DataFrame(results, columns=['(p,d,q)', '(P,D,Q,s)', '평균_RMSE']).sort_values('평균_RMSE')
print("\n===== 모델별 평균 RMSE 비교 =====")
print(results_df)


## SARIMAX(외생변수 추가 -> 휴일개수, 추석, 설여부, 직전년도 기온)

In [ ]:
df_temp = pd.read_csv('./수요예측부문 데이터 제공/csv파일 모음/기온 데이터.csv', encoding='cp949', skiprows=6)
# 1) 문자열 정리
df_temp['년월'] = df_temp['년월'].str.strip()  # 앞뒤 공백/탭 제거

# 2) datetime 변환
df_temp['년월'] = pd.to_datetime(df_temp['년월'], format='%Y-%m')

# 3) 불필요한 열 제거 (지점이 전부 '전국'이면 삭제)
df_temp = df_temp.drop(columns=['지점'])
df_temp = df_temp.iloc[:, [0,1,3]]  # '년월'과 '평균기온' 열만 남기기
df_temp = df_temp.rename(columns={'평균기온(℃)': '평균기온', '평균최고기온(℃)': '평균최고기온' })

# lag12
df_temp['평균기온_lag12'] = df_temp['평균기온'].shift(12)
df_temp['평균최고기온_lag12'] = df_temp['평균최고기온'].shift(12)
df_temp.dropna(inplace=True)
df_temp = df_temp.iloc[:,[0, 3,4]]
df_temp.reset_index(drop=True, inplace=True)
df_temp.head()

df2 = pd.merge(df, df_temp, left_index=True, right_on='년월', how='left').set_index('년월')
df2.head()

In [ ]:
# 공휴일 변수

hday = pd.read_csv('./수요예측부문 데이터 제공/csv파일 모음/외생변수/공휴일 캘린더.csv', encoding='utf-8')
hday = hday.iloc[:,[0,3]]
hday

data = {
    "year_month": ["2025-01", "2025-02", "2025-03"],      # 총 공휴일 수
    "holidays_weekday": [4, 0, 1],     # 평일 공휴일 수
}

df_holidays_2025Q1 = pd.DataFrame(data)

df_holidays_2025Q1

df_holiday = pd.concat([hday, df_holidays_2025Q1], axis=0).reset_index(drop=True)
df_holiday


# 설날/추석 실제 달력 기반 (주요일 기준 월만 표시)
seollal_months = [
    '2021-02','2022-02','2023-01','2024-02','2025-01','2025-01'  # 25년은 1/29~2/1이 설날 연휴
]
chuseok_months = [
    '2021-09','2022-09','2023-09','2024-09','2025-10'  # 25년 추석은 10/6
]

# 더미 생성
df_holiday['설날'] = df_holiday['year_month'].isin(seollal_months).astype(int)
df_holiday['추석'] = df_holiday['year_month'].isin(chuseok_months).astype(int)

df_holiday

df_holiday['year_month'] = pd.to_datetime(df_holiday['year_month'], format='%Y-%m')

df3 = pd.merge(df2, df_holiday, left_index=True, right_on='year_month', how='left').set_index('year_month')
df3.head()

In [ ]:
# 표준화
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

df3[[ '평균기온_lag12', '평균최고기온_lag12']] = scaler.fit_transform(df3[[ '평균기온_lag12', '평균최고기온_lag12']])
df3.head()

In [ ]:
sns.heatmap(df3.corr(), annot=True, fmt=".2f", cmap='coolwarm')
plt.title('Feature Correlation')
plt.show()

In [ ]:
# sarimax
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ===== 데이터 준비 =====
df = df3.asfreq('MS')

# 외생변수 정의
exog_vars = ['설날', '추석']

season_length = 12   # 월별 계절 주기
folds = 5     # 교차검증 fold 개수
h = 3                # 한 번에 예측할 기간 (3개월 ahead)

rmse_list = []
pred_df_list = []

# ===== Rolling Forecast Cross Validation =====
for i in range(1, folds + 1):
    split_point = len(df) - (folds - i) * h
    train = df.iloc[:split_point - h]
    test = df.iloc[split_point - h:split_point]

    if len(train) < season_length:
        continue

    # 외생변수 분리
    train_exog = train[exog_vars]
    test_exog = test[exog_vars]

    # SARIMAX 모델 정의
    model = SARIMAX(
        train['판매수량'],
        exog=train_exog,
        order=(0, 1, 1),
        seasonal_order=(0, 1, 1, 12),
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    result = model.fit(disp=False)

    # 3개월 ahead 예측
    forecast = result.forecast(steps=h, exog=test_exog)
    forecast = pd.Series(forecast, index=test.index, name=f'fold{i}_예측')

    rmse = np.sqrt(mean_squared_error(test['판매수량'], forecast))
    rmse_list.append(rmse)
    pred_df_list.append(pd.DataFrame({'실제': test['판매수량'], '예측': forecast}))

    print(f"Fold {i}: Train={train.index[0].strftime('%Y-%m')}~{train.index[-1].strftime('%Y-%m')}, "
          f"Test={test.index[0].strftime('%Y-%m')}~{test.index[-1].strftime('%Y-%m')}, RMSE={rmse:,.0f}")

# ===== 결과 요약 =====
print("\n📊 평균 RMSE:", round(np.mean(rmse_list), 2))

# ===== 시각화 =====
plt.figure(figsize=(12,6))
plt.plot(df.index, df['판매수량'], label='Actual', color='black')

for i, pred in enumerate(pred_df_list):
    plt.plot(pred.index, pred['예측'], label=f'Fold {i+1} Forecast', linestyle='--')

plt.title('SARIMAX(1,1,1)(0,1,1,12) Rolling Forecast (3개월 단위, 10-Fold)')
plt.xlabel('날짜')
plt.ylabel('판매수량')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()



In [ ]:
model = SARIMAX(train['판매수량'], exog=train[exog_vars],
                order=(0,1,1), seasonal_order=(0,1,1,12))
result = model.fit(disp=False)
print(result.summary())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings('ignore')

# ===== 데이터 준비 =====
df = df3.asfreq('MS')

exog_vars = ['평균기온_lag12', '평균최고기온_lag12', 'holidays_weekday', '설날', '추석']

# ===== 파라미터 설정 =====
season_length = 12
folds = 5
h = 3

# ===== AR(p), SAR(P)만 바꿔서 탐색 =====
p_values = [0, 1]
P_values = [0, 1]
results = []

for p_ in p_values:
    for P_ in P_values:
        order = (p_, 1, 1)
        seasonal_order = (P_, 1, 1, 12)

        rmse_list = []

        print(f"\n===== 모델: SARIMAX{order}x{seasonal_order} =====")

        for i in range(1, folds + 1):
            split_point = len(df) - (folds - i) * h
            train = df.iloc[:split_point - h]
            test = df.iloc[split_point - h:split_point]

            if len(train) < season_length:
                continue

            train_exog = train[exog_vars]
            test_exog = test[exog_vars]

            try:
                model = SARIMAX(
                    train['판매수량'],
                    exog=train_exog,
                    order=order,
                    seasonal_order=seasonal_order,
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )
                result = model.fit(disp=False)
                forecast = result.forecast(steps=h, exog=test_exog)

                rmse = np.sqrt(mean_squared_error(test['판매수량'], forecast))
                rmse_list.append(rmse)
            except Exception as e:
                print(f"⚠️ Fold {i} 실패 ({e})")
                continue

        if rmse_list:
            avg_rmse = np.mean(rmse_list)
            results.append((order, seasonal_order, avg_rmse))
            print(f"📊 평균 RMSE = {avg_rmse:,.2f}")

# ===== 결과 정리 =====
results_df = pd.DataFrame(results, columns=['(p,d,q)', '(P,D,Q,s)', '평균_RMSE']).sort_values('평균_RMSE')
print("\n===== 모델별 평균 RMSE 비교 =====")
print(results_df)


# 물류센터A 신선식품

In [ ]:
df_fresh = pd.read_csv('./수요예측부문 데이터 제공/csv파일 모음/A물류센터 매출_2021~2024_신선식품_전치리.csv')
df_fresh_time = df_fresh.groupby('연월')[['판매수량']].sum().reset_index()
df_fresh_time['연월'] = pd.to_datetime(df_fresh_time['연월'].astype(str))
df_fresh_time.set_index('연월', inplace=True)
# acf, pacf

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# 1행 2열로 시각화
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plot_acf(df_fresh_time['판매수량'], lags=24, ax=plt.gca())
plt.title("ACF")
plt.subplot(1,2,2)
plot_pacf(df_fresh_time['판매수량'], lags=24, ax=plt.gca())
plt.title("PACF")
plt.tight_layout()
plt.show()

## SARIMA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings("ignore")

# ===== 데이터 준비 =====


df = df_fresh_time.asfreq('MS')

# ===== 파라미터 설정 =====
season_length = 12   # 월별 데이터니까 12개월 주기
folds = 5          # 교차검증 fold 개수
h = 3                # 한 번에 예측할 기간 (3개월 ahead)

rmse_list = []
pred_df_list = []    # fold별 예측값 저장용

# ===== Rolling CV =====
for i in range(1, folds+1):
    split_point = len(df) - (folds - i)*h
    train = df.iloc[:split_point - h]
    test = df.iloc[split_point - h:split_point]

    if len(train) < season_length:
        continue

    # SARIMA(1,1,1)(0,1,1,12) 모델
    model = SARIMAX(train['판매수량'],
                    order=(1,1,1),
                    seasonal_order=(0,1,1,12),
                    enforce_stationarity=False,
                    enforce_invertibility=False)
    result = model.fit(disp=False)

    # 3개월 ahead 예측
    forecast = result.forecast(steps=h)
    forecast = pd.Series(forecast, index=test.index, name=f'fold{i}_예측')

    rmse = np.sqrt(mean_squared_error(test['판매수량'], forecast))
    rmse_list.append(rmse)
    pred_df_list.append(pd.DataFrame({'실제': test['판매수량'], '예측': forecast}))

    print(f"Fold {i}: Train={train.index[0].strftime('%Y-%m')}~{train.index[-1].strftime('%Y-%m')}, "
          f"Test={test.index[0].strftime('%Y-%m')}~{test.index[-1].strftime('%Y-%m')}, RMSE={rmse:,.0f}")

# ===== 결과 요약 =====
print("\n📊 평균 RMSE:", round(np.mean(rmse_list), 2))

# ===== 시각화 =====
plt.figure(figsize=(12,6))
plt.plot(df.index, df['판매수량'], label='Actual', color='black')

# 각 fold별 예측 구간 시각화
for i, pred in enumerate(pred_df_list):
    plt.plot(pred.index, pred['예측'], label=f'Fold {i+1} Forecast', linestyle='--')

plt.title('SARIMA(1,1,1)(0,1,1,12) Rolling Forecast (3개월 단위, 10-Fold)')
plt.xlabel('날짜')
plt.ylabel('판매수량')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 파라미터 튜닝

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

import warnings
warnings.filterwarnings('ignore')

# ===== 데이터 준비 =====

df = df.asfreq('MS')

# ===== 파라미터 설정 =====
season_length = 12   # 월별 데이터니까 12개월 주기
folds = 5           # 교차검증 fold 개수
h = 3                # 한 번에 예측할 기간 (3개월 ahead)

import itertools

# 후보 파라미터 조합
param_list = [
    ((1,1,1), (0,1,1,12)),
    ((0,1,1), (0,1,1,12)),
    ((1,1,0), (0,1,1,12)),
    ((2,1,1), (1,1,0,12)),
]

results = []

for (order, seasonal_order) in param_list:
    rmse_list = []
    print(f"\n===== 모델: SARIMA{order}x{seasonal_order} =====")

    for i in range(1, folds+1):
        split_point = len(df) - (folds - i)*h
        train = df.iloc[:split_point - h]
        test = df.iloc[split_point - h:split_point]

        if len(train) < season_length:
            continue

        model = SARIMAX(train['판매수량'],
                        order=order,
                        seasonal_order=seasonal_order,
                        enforce_stationarity=False,
                        enforce_invertibility=False)
        result = model.fit(disp=False)

        forecast = result.forecast(steps=h)
        rmse = np.sqrt(mean_squared_error(test['판매수량'], forecast))
        rmse_list.append(rmse)

    avg_rmse = np.mean(rmse_list)
    results.append((order, seasonal_order, avg_rmse))
    print(f"📊 평균 RMSE = {avg_rmse:,.2f}")

# 결과를 표로 정리
results_df = pd.DataFrame(results, columns=['(p,d,q)', '(P,D,Q,s)', '평균_RMSE']).sort_values('평균_RMSE')
print("\n===== 모델별 평균 RMSE 비교 =====")
print(results_df)


## seasonal naive

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt



df = df_fresh_time.asfreq('MS')
# ===== 파라미터 설정 =====
season_length = 12   # 월별 데이터니까 12개월 주기
folds = 5       # 교차검증 fold 개수
h = 3                # 한 번에 예측할 기간 (3개월 ahead)

rmse_list = []
pred_df_list = []    # fold별 예측값 저장용

# ===== Rolling CV =====
for i in range(1, folds+1):
    split_point = len(df) - (folds - i)*h
    train = df.iloc[:split_point - h]
    test = df.iloc[split_point - h:split_point]

    if len(train) < season_length:
        continue

    # Seasonal naive: 작년 같은 달의 값을 그대로 복제
    forecast = []
    for j in range(h):
        idx = -season_length + j  # 작년 동일 월 기준
        forecast.append(train['판매수량'].iloc[idx])
    forecast = pd.Series(forecast, index=test.index, name=f'fold{i+1}_예측')

    rmse = np.sqrt(mean_squared_error(test['판매수량'], forecast))
    rmse_list.append(rmse)
    pred_df_list.append(pd.DataFrame({'실제': test['판매수량'], '예측': forecast}))

    print(f"Fold {i}: Train={train.index[0].strftime('%Y-%m')}~{train.index[-1].strftime('%Y-%m')}, "
          f"Test={test.index[0].strftime('%Y-%m')}~{test.index[-1].strftime('%Y-%m')}, RMSE={rmse:,.0f}")

# ===== 결과 요약 =====
print("\n📊 평균 RMSE:", round(np.mean(rmse_list), 2))

# ===== 시각화 =====
plt.figure(figsize=(12,6))
plt.plot(df.index, df['판매수량'], label='Actual', color='black')

# 각 fold별 예측 구간 시각화
for i, pred in enumerate(pred_df_list):
    plt.plot(pred.index, pred['예측'], label=f'Fold {i+1} Forecast', linestyle='--')

plt.title('Seasonal Naive Rolling Forecast (3개월 단위, 10-Fold)')
plt.xlabel('날짜')
plt.ylabel('판매수량')
plt.legend()
plt.grid(True)
plt.tight_layout()

# ===== 그래프 저장 =====

plt.show()


In [ ]:
df2 = pd.merge(df_fresh_time, df_temp, left_index=True, right_on='년월', how='left').set_index('년월')
df3 = pd.merge(df2, df_holiday, left_index=True, right_on='year_month', how='left').set_index('year_month')
df3.head()

## SARIMAX(외생변수추가)

In [ ]:
# sarimax
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ===== 데이터 준비 =====
df = df3.asfreq('MS')

# 외생변수 정의
exog_vars = ['평균기온_lag12','holidays_weekday', '설날', '추석']

season_length = 12   # 월별 계절 주기
folds = 5     # 교차검증 fold 개수
h = 3                # 한 번에 예측할 기간 (3개월 ahead)

rmse_list = []
pred_df_list = []

# ===== Rolling Forecast Cross Validation =====
for i in range(1, folds + 1):
    split_point = len(df) - (folds - i) * h
    train = df.iloc[:split_point - h]
    test = df.iloc[split_point - h:split_point]

    if len(train) < season_length:
        continue

    # 외생변수 분리
    train_exog = train[exog_vars]
    test_exog = test[exog_vars]

    # SARIMAX 모델 정의
    model = SARIMAX(
        train['판매수량'],
        exog=train_exog,
        order=(0, 1, 1),
        seasonal_order=(0, 1, 1, 12),
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    result = model.fit(disp=False)

    # 3개월 ahead 예측
    forecast = result.forecast(steps=h, exog=test_exog)
    forecast = pd.Series(forecast, index=test.index, name=f'fold{i}_예측')

    rmse = np.sqrt(mean_squared_error(test['판매수량'], forecast))
    rmse_list.append(rmse)
    pred_df_list.append(pd.DataFrame({'실제': test['판매수량'], '예측': forecast}))

    print(f"Fold {i}: Train={train.index[0].strftime('%Y-%m')}~{train.index[-1].strftime('%Y-%m')}, "
          f"Test={test.index[0].strftime('%Y-%m')}~{test.index[-1].strftime('%Y-%m')}, RMSE={rmse:,.0f}")

# ===== 결과 요약 =====
print("\n📊 평균 RMSE:", round(np.mean(rmse_list), 2))

# ===== 시각화 =====
plt.figure(figsize=(12,6))
plt.plot(df.index, df['판매수량'], label='Actual', color='black')

for i, pred in enumerate(pred_df_list):
    plt.plot(pred.index, pred['예측'], label=f'Fold {i+1} Forecast', linestyle='--')

plt.title('SARIMAX(1,1,1)(0,1,1,12) Rolling Forecast (3개월 단위, 10-Fold)')
plt.xlabel('날짜')
plt.ylabel('판매수량')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()



# 25년 1,2,3 예측값 산출

In [ ]:
# 생수, 음료, 건강 
# ===== 모델: SARIMA(1, 1, 1)x(0, 1, 1, 12) =====

df_bev = pd.read_csv('./수요예측부문 데이터 제공/csv파일 모음/A물류센터 매출_2021~2024_생수음료건강_월별판매수량.csv')

df_bev['연월'] = pd.to_datetime(df_bev['연월'])
df_bev.set_index('연월', inplace=True)
df_bev = df_bev.asfreq('MS')

model = SARIMAX(df_bev['판매수량'],
                order=(1,1,1),
                seasonal_order=(0,1,1,12),
                enforce_stationarity=False,
                enforce_invertibility=False)

# 3개월 예측

result = model.fit(disp=False)
forecast = result.get_forecast(steps=3)

pred_mean = forecast.predicted_mean
conf_int = forecast.conf_int()

# 5️⃣ 결과 출력
print("📅 예측 결과:")
print(pred_mean)
print("\n📈 신뢰구간:")
print(conf_int)

# 6️⃣ 시각화
plt.figure(figsize=(10,4))
plt.plot(df_bev.index, df_bev['판매수량'], label='Actual')
plt.plot(pred_mean.index, pred_mean, label='Forecast', color='red', marker='o')
plt.fill_between(conf_int.index,
                 conf_int.iloc[:,0],
                 conf_int.iloc[:,1],
                 color='pink', alpha=0.3)
plt.title('SARIMA 3-Month Forecast (생수/음료/건강)')
plt.legend()
plt.show()



In [ ]:
# 신선식품
# ===== 모델: SARIMA(1, 1, 1)x(0, 1, 1, 12) =====

model = SARIMAX(df_fresh_time['판매수량'],
                order=(1,1,1),
                seasonal_order=(0,1,1,12),
                enforce_stationarity=False,
                enforce_invertibility=False)

# 3개월 예측

result = model.fit(disp=False)
forecast = result.get_forecast(steps=3)

pred_mean = forecast.predicted_mean
conf_int = forecast.conf_int()

# 5️⃣ 결과 출력
print("📅 예측 결과:")
print(pred_mean)
print("\n📈 신뢰구간:")
print(conf_int)

# 6️⃣ 시각화
plt.figure(figsize=(10,4))
plt.plot(df_fresh_time.index, df_fresh_time['판매수량'], label='Actual')
plt.plot(pred_mean.index, pred_mean, label='Forecast', color='red', marker='o')
plt.fill_between(conf_int.index,
                 conf_int.iloc[:,0],
                 conf_int.iloc[:,1],
                 color='pink', alpha=0.3)
plt.title('SARIMA 3-Month Forecast (신선식품)')
plt.legend()
plt.show()